In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""PhotoBox: fixed ColorChecker ROIs, affine CCM, and pixelwise sample hue.

Jupyter: execute this entire script in a cell and select the input folder in the
dialog. If a native dialog is unavailable, enter the folder path at the prompt.
Jupyter kernel arguments (-f <connection file>) are removed automatically.
Install dependencies in Jupyter if needed: %pip install numpy opencv-python
Optional command-line use: python photobox_ccm_hue.py --input-dir <folder>
Optional: --output-dir <folder>. Default output: the ccm_hue_results subfolder
inside the selected input folder. Images directly in the input folder are
processed alphabetically; subfolders are not searched.

Measurement procedure
---------------------
* Original resolution: 5472 x 7296 px; no resizing or geometric rectification.
* 24 fixed ColorChecker ROIs from the user-specified grid.
* Per RGB channel: remove the k=floor(0.10*N) smallest and largest values;
  average the remaining N-2*k values (channelwise rank-based trimming).
* Unweighted least-squares fit: RGB_corrected = M @ RGB + offset.
  Fit the decoded 8-bit RGB values directly, without gamma linearization,
  weighting, or regularization. Reference values: HTML app v1.0.0.
* Apply the CCM to every image pixel, then clip the results to [0,255].
* Sample: convert each corrected pixel to HSV, then calculate the circular mean
  of defined hue values. No trimming is applied to the sample hue values.
* Calculate hue BEFORE rounding for the saved 8-bit PNG. Exactly achromatic
  pixels (S=0) are counted and excluded from the hue mean.

The combined sample_hue.csv contains exactly Image and Sample_Hue_deg.
A separate CCM is fitted for each image. Per-image outputs include a corrected
PNG, the CCM in NPZ format, ColorChecker measurements in CSV format, and
parameters/results in JSON format, stored in individual subfolders. Failed
images have an empty hue entry in the combined CSV. Errors are printed and
processing continues with the remaining images.
Python >=3.9; NumPy and OpenCV are the only required external dependencies.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np


# -------------------------- Settings --------------------------
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_WIDTH, EXPECTED_HEIGHT = 5472, 7296

# Centers of the four corner patches, using the user-specified grid labels.
CORNER_A1 = (1486, 3755)
CORNER_A6 = (3950, 3755)
CORNER_D1 = (1486, 5041)
CORNER_D6 = (3950, 5041)
PATCH_W, PATCH_H = 363, 299
FILL_X, FILL_Y = 0.75, 0.80
N_ROWS, N_COLS = 4, 6
TRIM_FRACTION = 0.10
CHUNK_ROWS = 256  # Memory management only; resolution and pixel values are preserved.

# Rectangles: (name, x1, y1, x2, y2); x2 and y2 are EXCLUSIVE bounds.
ROI = Tuple[str, int, int, int, int]
SAMPLE_ROI: ROI = ("Sample", 2050, 2480, 2475, 2900)

# Copied unchanged from handy_app_v1.0.0.html, in row-major order.
# User grid a1...a6 corresponds to app A1...F1; b1...b6 to A2...F2, etc.
REFERENCE_RGB = np.array([
    [116, 79, 65], [197, 144, 127], [91, 120, 155],
    [91, 108, 64], [131, 127, 175], [95, 189, 172],
    [224, 124, 48], [69, 90, 167], [197, 80, 95],
    [93, 58, 104], [156, 187, 58], [227, 161, 39],
    [40, 62, 145], [61, 147, 70], [178, 54, 57],
    [236, 199, 15], [191, 79, 146], [0, 133, 165],
    [241, 242, 235], [201, 202, 201], [161, 163, 163],
    [121, 121, 121], [83, 84, 85], [50, 50, 50],
], dtype=np.float64)
APP_PATCH_CODES = [f"{col}{row}" for row in range(1, 5) for col in "ABCDEF"]


def build_grid() -> List[ROI]:
    """Reproduce the specified grid by bilinearly interpolating patch centers."""
    half_w = int(round(FILL_X * PATCH_W / 2))
    half_h = int(round(FILL_Y * PATCH_H / 2))
    rois = []
    for r in range(N_ROWS):
        v = r / (N_ROWS - 1)
        for c in range(N_COLS):
            u = c / (N_COLS - 1)
            cx = ((1-u)*(1-v)*CORNER_A1[0] + u*(1-v)*CORNER_A6[0]
                  + (1-u)*v*CORNER_D1[0] + u*v*CORNER_D6[0])
            cy = ((1-u)*(1-v)*CORNER_A1[1] + u*(1-v)*CORNER_A6[1]
                  + (1-u)*v*CORNER_D1[1] + u*v*CORNER_D6[1])
            cx, cy = int(round(cx)), int(round(cy))
            rois.append((f"{'abcd'[r]}{c+1}",
                         cx-half_w, cy-half_h, cx+half_w, cy+half_h))
    return rois


def validate_rois(rois: List[ROI], width: int, height: int) -> None:
    for name, x1, y1, x2, y2 in rois:
        if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError(f"ROI {name} {(x1, y1, x2, y2)} is outside "
                             f"the image bounds ({width} x {height} px).")


def read_image(path: Path) -> np.ndarray:
    """Decode an 8-bit image without resizing or automatic EXIF rotation."""
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")
    # fromfile/imdecode also supports Windows paths containing non-ASCII characters.
    bgr = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    if bgr is None:
        raise ValueError(f"Image could not be decoded: {path}")
    if bgr.dtype != np.uint8 or bgr.ndim != 3:
        raise ValueError("Expected an 8-bit color image, such as PNG or JPEG.")
    if bgr.shape[2] == 4:
        if np.any(bgr[:, :, 3] != 255):
            raise ValueError("The image contains transparent pixels. Please use "
                             "a fully opaque RGB image.")
        bgr = bgr[:, :, :3].copy()
    if bgr.shape[2] != 3:
        raise ValueError("The image must have three color channels.")
    return bgr


def trimmed_rgb(bgr_roi: np.ndarray) -> np.ndarray:
    """Remove exactly the k lowest and k highest ranked values per channel."""
    pixels = bgr_roi[:, :, ::-1].reshape(-1, 3)  # BGR -> RGB
    n = pixels.shape[0]
    k = int(np.floor(TRIM_FRACTION * n))
    if not 0 <= TRIM_FRACTION < 0.5 or n == 0 or n - 2*k <= 0:
        raise ValueError("Invalid trimming fraction or empty ROI.")
    ordered = np.sort(pixels, axis=0)
    return ordered[k:n-k].mean(axis=0, dtype=np.float64)


def fit_ccm(measured: np.ndarray, reference: np.ndarray):
    """Fit unweighted least squares; M is oriented for column vectors."""
    if measured.shape != (24, 3) or reference.shape != (24, 3):
        raise ValueError("The fit requires exactly 24 RGB triplets.")
    design = np.column_stack([measured, np.ones(24)])
    coefficients, _, rank, _ = np.linalg.lstsq(design, reference, rcond=None)
    if rank < 4 or not np.all(np.isfinite(coefficients)):
        raise ValueError("The ColorChecker measurements do not define a unique "
                         "affine fit. Check the ROI positions and input image.")
    matrix = coefficients[:3, :].T
    offset = coefficients[3, :]
    return matrix, offset


def correct_image(bgr: np.ndarray, matrix: np.ndarray,
                  offset: np.ndarray, sample_roi: ROI):
    """Correct ALL original pixels; retain sample values before 8-bit rounding."""
    height, width = bgr.shape[:2]
    validate_rois([sample_roi], width, height)
    _, x1, y1, x2, y2 = sample_roi
    corrected_bgr = np.empty_like(bgr)
    sample_rgb = np.empty((y2-y1, x2-x1, 3), dtype=np.float32)
    for top in range(0, height, CHUNK_ROWS):
        bottom = min(height, top + CHUNK_ROWS)
        rgb = bgr[top:bottom, :, ::-1].astype(np.float64)
        # Array entries are RGB row vectors, hence the transpose M.T.
        corrected = rgb @ matrix.T + offset
        np.clip(corrected, 0.0, 255.0, out=corrected)
        overlap_top, overlap_bottom = max(top, y1), min(bottom, y2)
        if overlap_top < overlap_bottom:
            sample_rgb[overlap_top-y1:overlap_bottom-y1] = corrected[
                overlap_top-top:overlap_bottom-top, x1:x2]
        corrected_bgr[top:bottom] = np.rint(corrected[:, :, ::-1]).astype(np.uint8)
    return corrected_bgr, sample_rgb


def evaluate_sample(sample_rgb: np.ndarray) -> dict:
    """Convert RGB to HSV pixelwise, then calculate an unweighted circular hue mean."""
    rgb_01 = np.ascontiguousarray(sample_rgb, dtype=np.float32) / 255.0
    hsv = cv2.cvtColor(rgb_01, cv2.COLOR_RGB2HSV)
    hue, saturation, value = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    # Hue is undefined at S=0; no arbitrary saturation threshold is introduced.
    valid = (saturation > 0.0) & np.isfinite(hue)
    n_valid = int(np.count_nonzero(valid))
    if not n_valid:
        raise ValueError("The sample ROI contains no defined hue values (S=0).")
    angles = np.deg2rad(hue[valid].astype(np.float64))
    mean_cos = float(np.cos(angles).mean())
    mean_sin = float(np.sin(angles).mean())
    resultant = min(1.0, float(np.hypot(mean_cos, mean_sin)))
    if resultant < 1e-12:
        raise ValueError("The circular sample hue is numerically undefined "
                         "(resultant length close to zero).")
    mean_hue = float(np.degrees(np.arctan2(mean_sin, mean_cos)) % 360.0)
    return {
        "hue_mean_deg": mean_hue,
        "circular_resultant_length": resultant,
        "n_pixels_total": int(hue.size),
        "n_pixels_hue_defined": n_valid,
        "n_pixels_hue_undefined": int(hue.size - n_valid),
        "saturation_mean_percent": float(saturation.mean(dtype=np.float64) * 100),
        "value_mean_percent": float(value.mean(dtype=np.float64) * 100),
    }


def save_png(path: Path, bgr: np.ndarray) -> None:
    success, encoded = cv2.imencode(".png", bgr, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    if not success:
        raise OSError(f"PNG encoding failed: {path}")
    encoded.tofile(str(path))


def run(image_path: Path, output_dir: Optional[Path] = None,
        verbose: bool = True) -> dict:
    image_path = image_path.expanduser().resolve()
    bgr = read_image(image_path)
    height, width = bgr.shape[:2]
    if (width, height) != (EXPECTED_WIDTH, EXPECTED_HEIGHT):
        raise ValueError(f"Image size is {width} x {height} px; the fixed "
                         f"coordinates require {EXPECTED_WIDTH} x "
                         f"{EXPECTED_HEIGHT} px. Automatic resizing is disabled.")
    rois = build_grid()
    validate_rois(rois + [SAMPLE_ROI], width, height)
    measured = np.array([trimmed_rgb(bgr[y1:y2, x1:x2])
                         for _, x1, y1, x2, y2 in rois])
    matrix, offset = fit_ccm(measured, REFERENCE_RGB)
    fitted = np.clip(measured @ matrix.T + offset, 0.0, 255.0)
    fit_rmse = float(np.sqrt(np.mean((fitted - REFERENCE_RGB)**2)))
    if verbose:
        print(f"Image: {image_path}\nOriginal resolution: {width} x {height} px")
        print(f"ColorChecker: 24 ROIs, each {rois[0][3]-rois[0][1]} x "
              f"{rois[0][4]-rois[0][2]} px; 10% trimming at each tail per channel")
        print("Applying the affine CCM to the entire image ...", flush=True)
    corrected_bgr, sample_rgb = correct_image(bgr, matrix, offset, SAMPLE_ROI)
    sample_result = evaluate_sample(sample_rgb)

    output_dir = (output_dir.expanduser().resolve() if output_dir is not None
                  else image_path.parent / f"{image_path.stem}_ccm_hue")
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem
    files = {
        "corrected_image": output_dir / f"{stem}_corrected.png",
        "ccm": output_dir / f"{stem}_ccm.npz",
        "colorchecker_csv": output_dir / f"{stem}_colorchecker.csv",
        "result_json": output_dir / f"{stem}_result.json",
    }
    if any(path.resolve() == image_path for path in files.values()):
        raise ValueError("An output file would overwrite the input image.")
    save_png(files["corrected_image"], corrected_bgr)
    np.savez(files["ccm"], matrix=matrix, offset=offset,
             measured_rgb=measured, reference_rgb=REFERENCE_RGB,
             roi_xyxy=np.array([roi[1:] for roi in rois], dtype=np.int32))
    columns = ["roi", "app_code", "x1", "y1", "x2", "y2", "n_pixels",
               "n_trim_each_tail_per_channel", "measured_R", "measured_G",
               "measured_B", "reference_R", "reference_G", "reference_B"]
    with files["colorchecker_csv"].open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(columns)
        for i, (name, x1, y1, x2, y2) in enumerate(rois):
            n = (x2-x1)*(y2-y1)
            writer.writerow([name, APP_PATCH_CODES[i], x1, y1, x2, y2, n,
                             int(np.floor(TRIM_FRACTION*n)),
                             *measured[i].tolist(), *REFERENCE_RGB[i].astype(int).tolist()])
    report = {
        "input_image": str(image_path),
        "image_width_px": width, "image_height_px": height,
        "reference_source": "handy_app_v1.0.0.html: REFERENCE_CARD",
        "trim_fraction_each_tail_per_channel": TRIM_FRACTION,
        "ccm_equation": "RGB_corrected_column = matrix @ RGB_column + offset",
        "matrix": matrix.tolist(), "offset": offset.tolist(),
        "calibration_rgb_rmse_after_clipping": fit_rmse,
        "sample_roi_xyxy_exclusive": list(SAMPLE_ROI[1:]),
        "sample_method": "pixelwise HSV, then unweighted circular mean of defined hues",
        "hue_before_png_rounding": True,
        "sample": sample_result,
        "software": {"numpy": np.__version__, "opencv": cv2.__version__},
        "files": {key: str(path) for key, path in files.items()},
    }
    files["result_json"].write_text(
        json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8")

    if verbose:
        print("\nCCM (RGB order):\n", matrix)
        print("Offset:", offset)
        print(f"RGB RMSE on the 24 fitted patches: {fit_rmse:.6f} (no cross-validation)")
        print(f"\nSample-ROI: {SAMPLE_ROI[1:]}; "
              f"{sample_result['n_pixels_total']:,} pixels")
        print(f"Sample mean hue (circular, pixelwise HSV): "
              f"{sample_result['hue_mean_deg']:.6f} degrees")
        print(f"Defined hue: {sample_result['n_pixels_hue_defined']:,} pixels; "
              f"undefined hue: {sample_result['n_pixels_hue_undefined']:,} pixels")
        print("Hue is calculated before rounding for the 8-bit PNG.")
        print("\nSaved files:")
        for path in files.values():
            print(path)
    return report


def run_folder(input_dir: Path, output_dir: Optional[Path] = None) -> Path:
    """Process images directly in the folder; write a combined two-column CSV."""
    input_dir = input_dir.expanduser().resolve()
    if not input_dir.is_dir():
        raise NotADirectoryError(f"Input folder not found: {input_dir}")
    # Collect inputs BEFORE writing outputs; do not search subfolders.
    images = sorted((p for p in input_dir.iterdir()
                     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
                    key=lambda p: (p.name.casefold(), p.name))
    if not images:
        raise ValueError(f"No supported images found in folder: {input_dir}")
    output_dir = (output_dir.expanduser().resolve() if output_dir is not None
                  else input_dir / "ccm_hue_results")
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_csv = output_dir / "sample_hue.csv"
    failures = []
    print(f"Input folder: {input_dir}\nImages found: {len(images)}")
    # UTF-8 with BOM improves non-ASCII filename support in Windows/Excel.
    # CSV format: comma separator and decimal point.
    with summary_csv.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.writer(handle)
        writer.writerow(["Image", "Sample_Hue_deg"])
        handle.flush()
        for index, image_path in enumerate(images, start=1):
            print(f"[{index}/{len(images)}] {image_path.name} ...", flush=True)
            # Including the extension avoids collisions, e.g. Bio1.png and Bio1.jpg.
            image_output = output_dir / f"{image_path.name}_ccm_hue"
            try:
                report = run(image_path, image_output, verbose=False)
            except (OSError, ValueError, np.linalg.LinAlgError, cv2.error) as error:
                writer.writerow([image_path.name, ""])
                failures.append((image_path.name, str(error)))
                print(f"  ERROR: {error}", flush=True)
            else:
                hue = report["sample"]["hue_mean_deg"]
                writer.writerow([image_path.name, hue])
                print(f"  Sample hue: {hue:.6f} degrees", flush=True)
            handle.flush()  # Preserve completed rows if notebook execution is interrupted.
    print(f"\nSuccessfully processed: {len(images)-len(failures)}/{len(images)}")
    if failures:
        print("Failed images have an empty hue entry in the CSV:")
        for name, message in failures:
            print(f"  {name}: {message}")
    print(f"\nCombined sample hue CSV:\n{summary_csv}")
    return summary_csv


def prompt_input_directory() -> Optional[Path]:
    """Request a folder path when a native dialog is unavailable."""
    try:
        value = input("Input image folder (leave empty to cancel): ").strip()
    except EOFError as error:
        raise RuntimeError("Interactive input is unavailable. Supply the folder "
                           "with --input-dir or call run_folder(Path(...)).") from error
    # Accept paths pasted with matching surrounding single or double quotes.
    if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
        value = value[1:-1]
    return Path(value).expanduser() if value else None


def select_input_directory() -> Optional[Path]:
    """Open a folder picker, or fall back to a text prompt without a display."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError:
        print("Native folder dialog is unavailable. Enter the folder path below.")
        return prompt_input_directory()

    root = None
    try:
        root = tk.Tk()
        root.withdraw()
        try:
            root.attributes("-topmost", True)
        except tk.TclError:
            pass  # Some window managers do not support this attribute.
        selected = filedialog.askdirectory(
            parent=root, title="Select folder containing input images", mustexist=True)
    except tk.TclError:
        print("Native folder dialog is unavailable. Enter the folder path below.")
    else:
        # Cancelling a working dialog cancels processing; it does not open a prompt.
        return Path(selected).expanduser() if selected else None
    finally:
        if root is not None:
            try:
                root.destroy()
            except tk.TclError:
                pass
    return prompt_input_directory()


def main(argv=None) -> None:
    if argv is None and "ipykernel" in sys.modules:
        # Notebook cells inherit kernel arguments. Remove only connection-file
        # arguments; preserve explicit --input-dir/--output-dir options in %run.
        argv = []
        kernel_args = iter(sys.argv[1:])
        for argument in kernel_args:
            if argument in ("-f", "--f"):
                next(kernel_args, None)
            elif not argument.startswith(("-f=", "--f=")):
                argv.append(argument)
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--input-dir", "-i", type=Path, default=None,
                        help="Input image folder; omit to choose it interactively.")
    parser.add_argument("--output-dir", "-o", type=Path, default=None,
                        help="Output folder; default: <input folder>/ccm_hue_results.")
    args = parser.parse_args(argv)
    input_dir = args.input_dir if args.input_dir is not None else select_input_directory()
    if input_dir is None:
        print("Folder selection cancelled. No images were processed.")
        return
    run_folder(input_dir, args.output_dir)


if __name__ == "__main__":
    main()


Input folder: C:\Users\tnerger\Desktop\BOX_Auswertung\Box\neu\raw\well
Images found: 6
[1/6] cc_error_heatmap.png ...
  ERROR: Image size is 570 x 320 px; the fixed coordinates require 5472 x 7296 px. Automatic resizing is disabled.
[2/6] cc_hue_error_map_after.png ...
  ERROR: Image size is 480 x 320 px; the fixed coordinates require 5472 x 7296 px. Automatic resizing is disabled.
[3/6] cc_hue_error_map_before.png ...
  ERROR: Image size is 480 x 320 px; the fixed coordinates require 5472 x 7296 px. Automatic resizing is disabled.
[4/6] cc_hue_error_map_loocv.png ...
  ERROR: Image size is 480 x 320 px; the fixed coordinates require 5472 x 7296 px. Automatic resizing is disabled.
[5/6] korrigiert_IMG_20251022_163346.png ...
  Sample hue: 39.662255 degrees
[6/6] korrigiert_korrigiert_IMG_20251022_163346.png ...
  Sample hue: 39.662487 degrees

Successfully processed: 2/6
Failed images have an empty hue entry in the CSV:
  cc_error_heatmap.png: Image size is 570 x 320 px; the fixed coor